# MonopolyZero — ASU expert shard

Collects one seat-balanced ASU expert shard from an exact repository commit. For automated runs, upload `/content/monopolyzero-asu-job.json` before executing this notebook.

In [ ]:
from pathlib import Path
import hashlib, json, os, re, subprocess, sys, tarfile, urllib.request

CONTENT = Path(os.environ.get('MONOPOLYZERO_CONTENT', '/content'))
JOB = {
    'commit': '52b75d592df6e448d6de5afbd2ca5a360c61bc16',
    'shard': 'manual',
    'games': 32,
    'seed_base': 100000,
    'max_rounds': 200,
    'rollout_positions': 8,
    'workers': 1,
}
job_path = CONTENT / 'monopolyzero-asu-job.json'
if job_path.exists():
    JOB.update(json.loads(job_path.read_text()))
assert re.fullmatch(r'[0-9a-f]{40}', JOB['commit'])
assert re.fullmatch(r'[A-Za-z0-9_-]+', str(JOB['shard']))
assert min(int(JOB['games']), int(JOB['max_rounds'])) > 0
assert int(JOB['rollout_positions']) >= 0
assert int(JOB['workers']) in (1, 2)
assert int(JOB['games']) % (int(JOB['workers']) * 4) == 0
assert int(JOB['rollout_positions']) % int(JOB['workers']) == 0
OUTPUT = CONTENT / f"monopolyzero-asu-{JOB['shard']}.npz"
STATUS_PATH = CONTENT / f"monopolyzero-asu-{JOB['shard']}.json"
print(json.dumps(JOB, indent=2, sort_keys=True))

In [ ]:
ram_gib = os.sysconf('SC_PAGE_SIZE') * os.sysconf('SC_PHYS_PAGES') / 1024**3
STATUS = {**JOB, 'state': 'running', 'cpu_count': os.cpu_count(), 'ram_gib': round(ram_gib, 2)}
STATUS_PATH.write_text(json.dumps(STATUS, indent=2, sort_keys=True) + '\n')
print(f"CPU cores: {os.cpu_count()} | RAM: {ram_gib:.2f} GiB")

In [ ]:
archive_path = CONTENT / f"DeepRL_Monopoly-{JOB['commit']}.tar.gz"
urllib.request.urlretrieve(
    f"https://codeload.github.com/Darkosxl/DeepRL_Monopoly/tar.gz/{JOB['commit']}",
    archive_path,
)
with tarfile.open(archive_path, 'r:gz') as archive:
    members = archive.getmembers()
    roots = {Path(member.name).parts[0] for member in members if member.name}
    assert len(roots) == 1
    assert not any(member.name.startswith('/') or '..' in Path(member.name).parts for member in members)
    archive.extractall(CONTENT, filter='data')
REPOSITORY_ROOT = CONTENT / roots.pop()
assert (REPOSITORY_ROOT / 'monopoly_bench').is_dir()
print(f'Repository: {REPOSITORY_ROOT}')

In [ ]:
workers = int(JOB['workers'])
games_per_worker = int(JOB['games']) // workers
rollouts_per_worker = int(JOB['rollout_positions']) // workers
parts = [CONTENT / f"monopolyzero-asu-{JOB['shard']}-part-{index}.npz.part" for index in range(workers)]
commands = [[
    sys.executable, '-m', 'monopoly_bench', 'collect-asu',
    '--output', str(parts[index]),
    '--games', str(games_per_worker),
    '--seed-base', str(int(JOB['seed_base']) + index * games_per_worker),
    '--max-rounds', str(JOB['max_rounds']),
    '--rollout-positions', str(rollouts_per_worker),
] for index in range(workers)]
try:
    processes = [subprocess.Popen(command, cwd=REPOSITORY_ROOT) for command in commands]
    return_codes = [process.wait() for process in processes]
    for command, return_code in zip(commands, return_codes):
        if return_code:
            raise subprocess.CalledProcessError(return_code, command)
    sys.path.insert(0, str(REPOSITORY_ROOT))
    from monopoly_bench.training import load_asu_examples, save_asu_examples
    save_asu_examples(OUTPUT, load_asu_examples(parts))
    for part in parts:
        part.unlink(missing_ok=True)
except Exception as exc:
    STATUS.update(state='failed', error=f'{type(exc).__name__}: {exc}')
    raise
else:
    digest = hashlib.sha256(OUTPUT.read_bytes()).hexdigest()
    STATUS.update(state='complete', output=str(OUTPUT), sha256=digest)
finally:
    temporary = STATUS_PATH.with_suffix('.tmp')
    temporary.write_text(json.dumps(STATUS, indent=2, sort_keys=True) + '\n')
    temporary.replace(STATUS_PATH)
print(json.dumps(STATUS, indent=2, sort_keys=True))

In [ ]:
import numpy as np
with np.load(OUTPUT, allow_pickle=False) as shard:
    print({
        'positions': len(shard['states']),
        'rollout_positions': int(shard['teachers'].sum()),
        'state_shape': shard['states'].shape,
        'mask_shape': shard['legal_masks'].shape,
        'ruleset': str(shard['ruleset'].item()),
    })